# Brazilian E-Commerce Analysis — Schema Exploration

This notebook loads the Olist dataset, inspects all tables, documents table grain, and prepares the SQLite database for the SQL business analysis.

The main goal here is not to answer business questions yet. The goal is to understand the structure of the data first, because this dataset contains multiple related tables and careless joins can easily create misleading metrics.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DB_PATH = PROCESSED_DIR / "olist.db"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

In [ ]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews,
    "geolocation": geolocation,
    "category_translation": category_translation
}

## 1. Table Overview

The goal of this section is to inspect the size of each table and check for full duplicate rows.

In [ ]:
overview = []

for name, df in tables.items():
    overview.append({
        "table_name": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum()
    })

overview_df = pd.DataFrame(overview)
overview_df

### Table Overview Observations

- Most tables do not contain full duplicate rows.
- The geolocation table contains a large number of duplicate rows. This is expected because location records can repeat for the same zip-code prefix, city, or coordinate combination.
- The orders, customers, products, sellers, payments, reviews, and order_items tables appear structurally clean from a full-row duplicate perspective.
- Full-row duplicate checks are only a first pass. Key-level duplicate checks are still needed because “no full duplicate rows” does not guarantee that identifiers are unique.

## 2. Missing Values Overview

This section checks missing values by table and by column. Missing values are inspected separately from the table overview because they require more detailed column-level analysis.

In [ ]:
missing_summary = []

for name, df in tables.items():
    missing_counts = df.isna().sum()
    missing_percentages = (missing_counts / len(df) * 100).round(2)

    for column in df.columns:
        if missing_counts[column] > 0:
            missing_summary.append({
                "table_name": name,
                "column_name": column,
                "missing_count": missing_counts[column],
                "missing_percentage": missing_percentages[column]
            })

missing_summary_df = pd.DataFrame(missing_summary)
missing_summary_df.sort_values(
    by=["table_name", "missing_count"],
    ascending=[True, False]
)

## 3. Key Uniqueness Checks

This section checks whether important identifier columns behave like unique keys.

In [ ]:
key_uniqueness_checks = {
    "customers.customer_id": customers["customer_id"].is_unique,
    "customers.customer_unique_id": customers["customer_unique_id"].is_unique,
    "orders.order_id": orders["order_id"].is_unique,
    "products.product_id": products["product_id"].is_unique,
    "sellers.seller_id": sellers["seller_id"].is_unique,
    "reviews.review_id": reviews["review_id"].is_unique,
}

key_uniqueness_df = pd.DataFrame(
    key_uniqueness_checks.items(),
    columns=["key_column", "is_unique"]
)

key_uniqueness_df

### Key Uniqueness Observation

- `customer_id` is unique, while `customer_unique_id` is not. This means that `customer_id` identifies a specific order/customer record, while `customer_unique_id` can appear across multiple purchases by the same real customer. These repeated values should not be dropped because they may be useful for repeat-customer analysis.
- `review_id` is not unique, so it should not be assumed to be a primary key by itself. Before cleaning, duplicated review IDs need to be inspected to understand whether they represent repeated review records, multiple review events, or a composite-key situation.

## 4. Duplicate Key Counts

This section counts duplicated values in important key columns. This is more informative than only checking whether a column is unique.

In [ ]:
key_checks = {
    "customers.customer_id": customers["customer_id"].duplicated().sum(),
    "customers.customer_unique_id": customers["customer_unique_id"].duplicated().sum(),
    "orders.order_id": orders["order_id"].duplicated().sum(),
    "products.product_id": products["product_id"].duplicated().sum(),
    "sellers.seller_id": sellers["seller_id"].duplicated().sum(),
    "reviews.review_id": reviews["review_id"].duplicated().sum(),
    "reviews.order_id": reviews["order_id"].duplicated().sum(),
}

key_checks_df = pd.DataFrame(
    key_checks.items(),
    columns=["key_column", "duplicated_values"]
)

key_checks_df

### Key Uniqueness Observations

* `customer_id`, `order_id`, `product_id`, and `seller_id` do not contain duplicated values in their respective tables, so they appear to behave like unique identifiers.
* `customer_unique_id` contains duplicated values. This is expected because it represents the same real customer across multiple purchases, while `customer_id` identifies a specific customer/order record.
* `review_id` and `reviews.order_id` both contain duplicated values, so the reviews table needs inspection before assuming a single-row-per-order or single-row-per-review structure.

## 5. Composite Key Checks

Some tables are not expected to have a single-column primary key. For these tables, composite keys are checked instead.

In [ ]:
composite_key_checks = {
    "order_items.order_id + order_item_id": order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum(),

    "payments.order_id + payment_sequential": payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum(),

    "reviews.review_id + order_id": reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum(),
}

composite_key_checks_df = pd.DataFrame(
    composite_key_checks.items(),
    columns=["composite_key", "duplicated_rows"]
)

composite_key_checks_df

### Composite Key Observations

* The `order_items` table has a unique `order_id + order_item_id` combination, which suggests that each row represents one item within an order.
* The `payments` table has a unique `order_id + payment_sequential` combination, which means that each row represents one payment record or payment sequence for an order.
* The `reviews` table has a unique `review_id + order_id` combination, even though `review_id` and `order_id` are not unique individually. This means the reviews table should not be treated as one-row-per-review-id or one-row-per-order without further care.

## 6. Table Grain Summary

Understanding table grain is one of the most important parts of this project. Each table is listed below with what one row represents.

- `customers`: one row represents one customer/order-level customer record. `customer_id` is unique, while `customer_unique_id` can repeat across multiple purchases by the same real customer.

- `orders`: one row represents one order. `order_id` is unique.

- `order_items`: one row represents one item inside an order. An order can have multiple item rows, so the natural row identifier is `order_id + order_item_id`.

- `products`: one row represents one product. `product_id` is unique.

- `sellers`: one row represents one seller. `seller_id` is unique.

- `payments`: one row represents one payment record or payment sequence for an order. An order can have multiple payment rows, so the natural row identifier is `order_id + payment_sequential`.

- `reviews`: one row represents a review record linked to an order. `review_id` and `order_id` are not unique individually, but `review_id + order_id` is unique.

- `geolocation`: one row represents a zip-code/location record. This table contains many duplicate rows and will require separate handling if used for location analysis.

- `category_translation`: one row represents a product category name and its English translation.

## 7. Data Type and Date Column Inspection

This section inspects date columns in the orders table. These columns will be important later for order trends, delivery-time calculations, and late-delivery analysis.

In [ ]:
orders.info()

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns].isna().sum()

In [ ]:
for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

orders[date_columns].dtypes

### Date Column Observations

- All main order timeline columns were successfully converted to datetime format.
- `order_purchase_timestamp` and `order_estimated_delivery_date` have no missing values.
- `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` contain missing values.
- Missing approval or delivery timestamps likely correspond to orders that were canceled, unavailable, or never completed.
- These columns will be useful later for delivery-time, delay, and order trend analysis.

## 8. Delivery Date Missingness by Order Status

Before calculating delivery time, missing delivery dates need to be inspected by order status.

In [ ]:
orders["order_status"].value_counts()

In [ ]:
orders.loc[
    orders["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()

### Delivery Date Missingness Observation

Most missing customer delivery dates belong to orders that were not completed, such as shipped, canceled, unavailable, invoiced, or processing orders. This suggests that missing delivery dates are mostly meaningful and related to order status rather than random missing data.

However, a small number of orders are marked as delivered while still missing `order_delivered_customer_date`. These rows may require handling later when calculating delivery-time metrics.

## 9. SQLite Database Setup

The cleaned table objects are written into a local SQLite database. This database is used in the SQL business analysis notebook.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

In [ ]:
for table_name, df in tables.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)

In [ ]:
pd.read_sql_query("""
SELECT COUNT(*) AS total_orders
FROM orders;
""", conn)

In [ ]:
pd.read_sql_query("""
SELECT 
    order_status,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
""", conn)

### SQL Setup Observation

The Olist tables were successfully loaded into SQLite. The first SQL checks confirm that the `orders` table can be queried from the notebook and that order status counts match the expected dataset structure.

## Schema Exploration Summary

This notebook confirmed the main structure of the Olist dataset and prepared the SQLite database for analysis.

The most important takeaways are:

- `orders`, `customers`, `products`, and `sellers` have clear unique identifiers.
- `order_items`, `payments`, and `reviews` require composite-key thinking.
- `customer_unique_id` can repeat because it represents the same real customer across multiple purchases.
- The geolocation table contains many duplicate rows and should be handled carefully if used later.
- Missing delivery timestamps are mostly connected to order status rather than random missingness.
- The SQLite database is ready for the main SQL business analysis.